In [ ]:
# In colab run this cell first to setup the file structure!
%cd /content
!rm -rf MOL518-Intro-to-Data-Analysis

!git clone https://github.com/shaevitz/MOL518-Intro-to-Data-Analysis.git
%cd MOL518-Intro-to-Data-Analysis/Precept_6

# BPY518 Precept : Kymographs

## What we'll do today

- Quick tour: the mother machine and why 1D-constrained movies help us
- Load a real published time-lapse movie into a NumPy stack
- Crop a single growth channel
- Threshold it to see the cells
- Build a **kymograph**: one image that summarizes the whole movie
- Heuristically segment and extract some data about cell division times

## The mother machine


> Wang, Robert, Pelletier, Dang, Taddei, Wright, Jun. *Robust growth of Escherichia coli*. **Current Biology** 20, 1099–1103 (2010). [doi:10.1016/j.cub.2010.04.045](https://doi.org/10.1016/j.cub.2010.04.045)

A microfluidic device with hundreds to thousands of narrow dead-end channels perpendicular to a flow trench. Bacteria loaded into a channel are trapped in a line. The cell at the closed end (the "mother") stays put, daughters get pushed out.

Importantly: **the growth is one-dimensional**. If we want to know how cells grow and divide, we only need one axis of position + time. That's a 2D image, known as a kymograph. Check Fig. 1 panel D of the paper.

## The movie

We will use **Supplementary Movie S1B** from the original paper.
An MP4 is just a compressed stack of images. We'll use `imageio` to turn it into a 3D NumPy array of shape `(T, Y, X)`

In [ ]:
# First we download the movie from the paper
import urllib.request, os

URL = "https://ars.els-cdn.com/content/image/1-s2.0-S0960982210005245-mmc3.mp4"
DEST = "wang2010_movieS1B.mp4"

if not os.path.exists(DEST):
    req = urllib.request.Request(
        URL,
        headers={
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
            "Referer": "https://www.sciencedirect.com/",
        },
    )
    with urllib.request.urlopen(req) as r, open(DEST, "wb") as f:
        f.write(r.read())

# Sanity check
size_mb = os.path.getsize(DEST) / 1e6
print(f"{DEST}: {size_mb:.1f} MB")
assert size_mb > 1, "Download too small — likely got an HTML error page"

If the cell above runs and you have the movie downloaded on colab, great! If something went wrong, you can manually download the MP4 into your PC and upload it on the colab notebook, renaming it to 'wang2010_movieS1B.mp4' (sorry to make you do this, I don't want Elsevier knocking on my door)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from scipy import ndimage

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.cmap'] = 'gray'

# Read every frame; imageio returns (T, Y, X, C) for color video.
movie = iio.imread('wang2010_movieS1B.mp4')
print('Raw shape:', movie.shape, 'dtype:', movie.dtype)

# The movie is grayscale saved as RGB, but in microscopy we really only have one channel (usually)
movie = movie[..., 0]

# Convert to float for math later.
movie = movie.astype(float)
print('Stack shape (T, Y, X):', movie.shape)


- **Time between frames:** 1 minute
- **Pixel size:** ~0.065 µm/pixel (guess based on other reported setups, 100× oil)

In [ ]:
DT_MIN = 1.0          # minutes per frame
PIXEL_UM = 0.065      # micrometers per pixel ()

n_frames, height, width = movie.shape
duration_min = n_frames * DT_MIN
print(f'Movie: {n_frames} frames = {duration_min:.0f} min '
      f'({duration_min/60:.1f} hours)')
print(f'Frame size: {height} x {width} px '
      f'= {height*PIXEL_UM:.1f} x {width*PIXEL_UM:.1f} µm')

## Looking at frames

We slice the stack and display three frames spread over time.

In [ ]:
first = 0
middle = n_frames // 2
last = n_frames - 1

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, t, label in zip(axes,
                        [first, middle, last],
                        ['First', 'Middle', 'Last']):
    ax.imshow(movie[t])
    ax.set_title(f'{label} frame (t = {t*DT_MIN:.0f} min)')
    ax.axis('off')
plt.tight_layout()
plt.show()

You should see a field of view with multiple vertical dark growth channels, cells visible as bright rods inside them.

## Picking one growth channel

For a kymograph we want to look at **one channel** from start to finish. Look at the frames above and pick a channel that stays occupied. Everything downstream uses a simple rectangular crop `stack[:, y0:y1, x0:x1]`.

In [ ]:
# y0, y1 define the vertical extent of the growth channel (closed end → open end).
# x0, x1 define its width.
y0, y1 = 0, 125
channel_width_px = 25
x0 = 205 # try different values here, channels are roughly 20 px apart
x1 = x0 + channel_width_px

channel = movie[:, y0:y1, x0:x1]
print('Single-channel stack shape:', channel.shape)

# Same sanity check, cropped.
fig, axes = plt.subplots(1, 3, figsize=(5, 6))
for ax, t, label in zip(axes,
                        [first, middle, last],
                        ['First', 'Middle', 'Last']):
    ax.imshow(channel[t])
    ax.set_title(f'{label}\nt = {t*DT_MIN:.0f} min')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Seeing the cells: a threshold

The cells are brighter than the background. From Lecture 31, a boolean comparison gives a binary mask. Look at the intensity histogram of one frame to choose a threshold.

In [ ]:
sample = channel[middle]

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(sample)
axes[0].set_title(f'Channel at t = {middle*DT_MIN:.0f} min')
axes[0].axis('off')

axes[1].hist(sample.ravel(), bins=50, color='cornflowerblue')
thresh = 0 # TODO set a value of the threshold so that the cells look approximately segmented on the right
axes[1].axvline(thresh, color='k', lw=2)
axes[1].text(thresh+5, axes[1].get_ylim()[1]*0.8, f'threshold = {thresh}')
axes[1].set_xlabel('Pixel intensity')
axes[1].set_ylabel('Count')

mask = sample > thresh
axes[2].imshow(mask)
axes[2].set_title('Binarized mask')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## From a 2D frame to a 1D line profile

The channel is narrow — ~25 pixels wide. Position along the channel is the only spatial axis we care about. We can collapse the channel width in two natural ways:

- **Line profile** through the middle column: `sample[:, sample.shape[1]//2]`
- **Mean across the channel width:** `sample.mean(axis=1)`

The mean is usually better — it averages out noise. This is also a lightweight version of the 1D projection idea from Lecture 31.

In [ ]:
center_col = sample.shape[1] // 2 # look at the center line
line_center = sample[:, center_col]
line_mean = sample.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes[0].imshow(sample)
axes[0].axvline(center_col, color='red', lw=1, ls='--')
axes[0].set_title('Channel with midline')
axes[0].axis('off')

y_px = np.arange(sample.shape[0])
y_um = y_px * PIXEL_UM
axes[1].plot(line_center, y_um, color='red', alpha=0.6, label='center column')
axes[1].plot(line_mean, y_um, color='black', lw=2, label='mean across width')
axes[1].invert_yaxis()   # match image orientation
axes[1].set_xlabel('Pixel intensity')
axes[1].set_ylabel('Position along channel (µm)')
axes[1].legend()
plt.tight_layout()
plt.show()

Each bright bump in the line profile is a cell. The flat dark stretches are the gaps between cells (or the ends of the channel).

## Building a kymograph

Now we do this for **every time point** and stack the results.

Our `channel` stack has shape `(T, Y, X)`. If we average across `X` (the channel width) we get an array of shape `(T, Y)` — one row per time point, each row is a 1D profile down the channel. Transposing gives us `(Y, T)`: **vertical axis = position along the channel, horizontal axis = time.**

That's the kymograph.

In [ ]:
kymo = channel.mean(axis=2).T   # shape (Y, T)
print('Kymograph shape (Y, T):', kymo.shape)

T_total = kymo.shape[1] * DT_MIN
Y_total = kymo.shape[0] * PIXEL_UM

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(kymo, aspect='auto',
          extent=[0, T_total, Y_total, 0])  # left, right, bottom, top
ax.set_xlabel('Time (min)')
ax.set_ylabel('Position along channel (µm)')
ax.set_title('Kymograph of a single growth channel')
plt.tight_layout()
plt.show()

Each bright band traveling left-to-right is one cell's life. The band at the very bottom of the channel (closed end) is the mother cell — it's continuous across the whole movie. Every division is a *fork*: one band splits, and the upper half starts moving upward as it gets pushed by the new mother below it. Cells that reach the open end disappear.

This is the same "feather" pattern as the Wang 2010 Fig. 1D.
If we zoom into a shorter interval, we can more clearly see the dynamics of cell divisions within the channel.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(kymo[:,:100], aspect='auto',
          extent=[0, 100, Y_total, 0])  # left, right, bottom, top
ax.set_xlabel('Time (min)')
ax.set_ylabel('Position along channel (µm)')
ax.set_title('Kymograph of a single growth channel (shorter timescale)')
plt.tight_layout()
plt.show()

## Exercise: estimating the mother cell's growth and division rate

Looking at the line profile plot we made earlier, each bright bump is a cell, and the **dips between bumps** mark the gaps between them. The mother cell sits at the closed end of the channel, so:

> *The mother cell occupies the stretch from the closed end to the first dip in the profile.*

So if we can find the position of that first dip at every time point, we have the mother's length as a function of time — without needing to use the full machinery of object segmentation and tracking, which will be the topic of later lectures. This is a form of axial or 1D segmentation, which in this case is facilitated a lot by the geometry of the channel.

A good tool for this is `scipy.signal.find_peaks`, which returns the positions of local maxima in a 1D array. To find *valleys* we just apply it to the negative of the profile:

```python
from scipy.signal import find_peaks
valleys, _ = find_peaks(-profile, distance=8, prominence=5)
```

The `distance` and `prominence` parameters tell it to ignore features that are too close together or too small to be real cell boundaries (you may need to tweak them).

#### Checking in one frame

For a single frame (here we pick the middle one), compute the mean-across-width profile, call `find_peaks` on its negative (so we find valleys!), and make a plot that overlays the detected valleys on the profile. Then identify which valley corresponds to the mother cell boundary, and report the mother cell length in µm.

In [ ]:
from scipy.signal import find_peaks

profile = channel[middle].mean(axis=1)
min_distance = 0.5 # um TODO set a typical distance between cells in microns
my_prominence = 4 # TODO set a value of prominence
valleys, _ = find_peaks(-profile, distance=int(min_distance / PIXEL_UM), prominence=my_prominence)

y_um = np.arange(len(profile)) * PIXEL_UM

## Plotting machinery

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(y_um, profile, color='black')
ax.plot(y_um[valleys], profile[valleys], 'rv', ms=10, label='valleys')
ax.set_xlabel('Position along channel (µm)')
ax.set_ylabel('Mean intensity')
ax.legend()
plt.tight_layout()
plt.show()


In the plot above, the red markers identifying valleys should match with the boundaries between cells! To find the mother cell's length, we just have to find the value between the bottom of the channel and the first valley above that (note that the y axis is 'inverted' such that the bottom of the channel is the highest value of y)

In [ ]:
# Mother cell = closed end (position 0) to the first valley
mother_end_px = valleys[-1]
mother_length_um = (y1 - mother_end_px) * PIXEL_UM
print(f'Estimated mother cell length at t = {middle} min: {mother_length_um:.2f} µm')

#### Computing the length at every frame

Repeat the measurement for every time point and plot mother cell length vs. time. Use a **log y-axis** — between divisions, exponential single-cell growth should appear as straight-line segments.

You should see a sawtooth: the length rises (growth), then halves (division), then rises again. This is exactly Fig. 1D, bottom panel, of Wang 2010, now made by you from the raw movie.

In [ ]:
lengths_um = np.zeros(n_frames)

for t in range(n_frames):
    profile = channel[t].mean(axis=1)
    valleys, _ = find_peaks(-profile,
                            distance=int(min_distance / PIXEL_UM),
                            prominence=my_prominence)
    if len(valleys) > 0:
        lengths_um[t] = (y1 - valleys[-1]) * PIXEL_UM
    else:
        lengths_um[t] = np.nan  # no valley found — channel empty or full

times_min = np.arange(n_frames) * DT_MIN

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(times_min[:], lengths_um[:], 'k.', ms=3)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Mother cell length (µm)')
ax.set_title('Mother cell length over time')
plt.tight_layout()
plt.show()


Now we can extract the intervals between consecutive divisions - which we extract by considering events where the mother's length abruptly drops. In this process we have to be careful, because we are doing using a very simple heuristic for segmentation it's important to filter out excessively short division times (here I chose < 10 min).

In [ ]:
# Compute frame-to-frame length differences
dL = np.diff(lengths_um)              # shape (n_frames - 1,)

# A division is a large negative jump.
# Threshold: drop of more than 1 µm in a single frame.
# (E. coli cells are ~1-3 µm; a halving is a ~0.5-1 µm drop.)
division_frames = np.where(dL < -0.5)[0] + 1   # +1 because diff shifts by one

# Keep only divisions that are at least min_gap_frames after the previous one
MIN_GENERATION_MIN = 10
min_gap_frames = int(MIN_GENERATION_MIN / DT_MIN)
filtered = [division_frames[0]]
for f in division_frames[1:]:
    if f - filtered[-1] >= min_gap_frames:
        filtered.append(f)
division_frames = np.array(filtered)

division_times = division_frames * DT_MIN

print(f'Found {len(division_frames)} putative divisions')
#print(f'Division times (min): {division_times}')

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(times_min, lengths_um, ms=3)
for t in division_times:
    ax.axvline(t, color='red', alpha=0.4, lw=1)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Mother cell length (µm)')
ax.set_title('Mother cell length with detected divisions')

Finally, we can look at our estimate of the distribution of division times! The figure reported in the paper is 20.9 ± 0.3 min, does our estimate at least match that?

In [ ]:
# Inter-division times = generation times
generation_times = np.diff(division_times)
print(f'Mean generation time: {np.nanmean(generation_times):.1f} min')
print(f'Std. deviation of the generation time: {np.nanstd(generation_times):.1f} min')

# Let's histogram the division time distribution
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(generation_times, bins=20, color='cornflowerblue')
ax.set_xlabel('Generation time (min)')